# Qwen3-TTS on Google Colab

Run Qwen3-TTS with CUDA GPU acceleration on Google Colab.

**Requirements:** A Colab runtime with GPU (T4 or better).

**Setup:** Upload the `Qwen3-TTS_UserFiles` folder to Google Drive at `My Drive/Qwen3-TTS_UserFiles/`, then run all cells in order.

In [ ]:
# === Settings (edit these before running!) === #@title Settings { display-mode: "form" }
MODEL_SIZE = "1.7B"  # @param ["1.7B", "0.6B"]
PRELOAD_CLONE = True    # @param {type:"boolean"}
PRELOAD_DESIGN = False  # @param {type:"boolean"}
PRELOAD_CUSTOM = False  # @param {type:"boolean"}
PRELOAD_ASR = False     # @param {type:"boolean"}
DEFAULT_MODE = "design"  # @param ["clone", "design", "custom"]
DEFAULT_VOICE_DESC = "A warm, friendly voice with clear articulation"  # @param {type:"string"}
MAX_NEW_TOKENS = 2048  # @param {type:"integer"}
MAX_CHUNK_CHARS = 500  # @param {type:"integer"}
COMPILE_MODEL = True  # @param {type:"boolean"}
AUTO_LAUNCH_UI = True  # @param {type:"boolean"}
USE_UV = True  # @param {type:"boolean"}
INSTALL_FLASH_ATTN = False  # @param {type:"boolean"}
FLASH_ATTN_VERSION = "2.7.4"  # @param {type:"string"}
TORCH_QUANTIZATION = "auto"  # @param ["auto", "none", "8bit", "4bit"]
AUDIO_LOADER = "torchaudio"  # @param ["torchaudio", "librosa"]
# Set INSTALL_FLASH_ATTN True on Ampere+ GPUs (L4, A100, H100) for ~5-10% speed gain.
# Pre-built wheels - no CUDA compilation.
# TORCH_QUANTIZATION="auto" picks 8bit on Turing (T4), none on Ampere+.

In [ ]:
# === GPU Detection ===
import torch

if not torch.cuda.is_available():
    print("WARNING: No GPU detected! TTS will be very slow on CPU.")
    print("Go to Runtime > Change runtime type > GPU")
else:
    gpu_name = torch.cuda.get_device_name(0)
    cap = torch.cuda.get_device_capability(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    cap_str = f"{cap[0]}.{cap[1]}"

    print(f"GPU: {gpu_name}")
    print(f"Compute Capability: {cap_str}")
    print(f"VRAM: {vram_gb:.1f} GB")
    print()

    if cap[0] >= 8:
        tier = "Ampere+"
        if INSTALL_FLASH_ATTN:
            print(f"Tier: {tier} — Flash Attention 2, bfloat16, torch.compile supported")
        else:
            print(f"Tier: {tier} — SDPA attention, bfloat16, torch.compile supported")
            print("  Set INSTALL_FLASH_ATTN = True in Settings for Flash Attention 2 (~5 min build)")
        print("Quantization: Not needed (full bf16 fits comfortably)")
        print(f"VRAM budget: ~3.5 GB per model, all 3 models fit in {vram_gb:.0f} GB")
    else:
        tier = "Turing"
        print(f"Tier: {tier} — SDPA attention, float16, 8-bit quantization")
        print("Quantization: 8-bit via bitsandbytes (~1.75 GB per model)")
        print(f"VRAM budget: ~1.75 GB per model, all 3 models fit in {vram_gb:.0f} GB")
        if COMPILE_MODEL:
            print()
            print("NOTE: torch.compile is less effective on Turing GPUs.")
            print("Consider setting COMPILE_MODEL = False in Settings.")

    print()
    print("Recommendation: Colab Pro with L4 GPU offers the best price/performance")
    print("(Flash Attention 2, bf16, ~0.74x RTF at 4.82 CU/hr).")

In [ ]:
# === Cell 1: Setup ===
import os
import json
import sys

# Mount Google Drive (force_remount picks up newly synced files on re-run)
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

PROJECT_DIR = '/content/drive/My Drive/Qwen3-TTS_UserFiles'
HOME_DIR = os.path.expanduser('~/Qwen3-TTS_UserFiles')

if not os.path.exists(PROJECT_DIR):
    raise FileNotFoundError(
        f"Project not found at '{PROJECT_DIR}'.\n"
        "Upload the Qwen3-TTS_UserFiles folder to 'My Drive/Qwen3-TTS_UserFiles/' in Google Drive."
    )

if not os.path.exists(HOME_DIR):
    os.symlink(PROJECT_DIR, HOME_DIR)
    print(f'Linked {PROJECT_DIR} -> {HOME_DIR}')

if HOME_DIR not in sys.path:
    sys.path.insert(0, HOME_DIR)

os.makedirs(os.path.expanduser('~/Downloads'), exist_ok=True)
!apt-get update -qq && apt-get install -y -qq ffmpeg > /dev/null

# ---------------------------------------------------------------------------
# Derive deps from pyproject.toml -- NEVER hand-curate (drift caused slowapi failure 2026-05-23)
# ---------------------------------------------------------------------------
try:
    import tomllib
except ModuleNotFoundError:
    import tomli as tomllib

PYPROJECT_PATH = os.path.join(HOME_DIR, "pyproject.toml")
REQUIRED_EXTRAS = ["torch", "server", "audio", "ui", "cuda", "rich"]

try:
    with open(PYPROJECT_PATH, "rb") as _f:
        _pyproject = tomllib.load(_f)
    _proj = _pyproject["project"]
    _deps = list(_proj.get("dependencies", []))
    _extras = _proj.get("optional-dependencies", {})
    for _ex in REQUIRED_EXTRAS:
        if _ex not in _extras:
            raise KeyError(f"pyproject.toml missing extra {_ex!r}")
        _deps.extend(_extras[_ex])
    _seen = set()
    DEPS_LIST = [d for d in _deps if not (d in _seen or _seen.add(d))]
    DEPS = " ".join(f'"{d}"' for d in DEPS_LIST)
    print(f"Derived {len(DEPS_LIST)} deps from pyproject.toml extras {REQUIRED_EXTRAS}")
except Exception as _e:
    print(f"WARN: could not parse pyproject.toml ({_e}). Using fallback.")
    DEPS = (
        '"slowapi>=0.1.9" "psutil>=5.9" "pyloudnorm>=0.1.1" "rich>=13.7.1" '
        '"fastapi>=0.110.0" "uvicorn[standard]>=0.29.0" "soundfile>=0.12.1" '
        '"torch>=2.0" "torchaudio>=2.0" "qwen-tts" "transformers>=4.57.3" '
        '"librosa>=0.10.1" "pyrubberband" "pydub" "scipy" "gradio>=5.0.0" '
        '"accelerate" "bitsandbytes" "click>=8.0" "pySBD>=0.3.4" '
        '"num2words>=0.5.13" "requests>=2.28.0" "numpy"'
    )

if USE_UV:
    !curl -LsSf https://astral.sh/uv/install.sh | sh
    os.environ["PATH"] += ":/root/.cargo/bin"
    UV_CACHE = "/root/.uv_cache"
    os.makedirs(UV_CACHE, exist_ok=True)
    os.environ["UV_CACHE_DIR"] = UV_CACHE
    import subprocess as _sp
    _uv_result = _sp.run(f"uv pip install --system {DEPS}", shell=True, stderr=_sp.PIPE, text=True)
    if _uv_result.returncode != 0:
        print(f"uv install failed (exit {_uv_result.returncode}), falling back to pip...\n{_uv_result.stderr[:2000]}")
        !pip install -q {DEPS}
    else:
        print("Packages installed via uv")
else:
    !pip install -q {DEPS}

# Verify every module imported at server module-scope
_VERIFY_IMPORTS = (
    "fastapi", "slowapi", "uvicorn", "click", "pyrubberband", "pysbd", "num2words",
    "psutil", "pyloudnorm", "librosa", "soundfile", "scipy", "pydub",
    "accelerate", "requests", "numpy", "torch", "torchaudio", "transformers",
    "gradio",
)
_missing = []
for _m in _VERIFY_IMPORTS:
    try:
        __import__(_m)
    except ImportError:
        _missing.append(_m)
if _missing:
    raise RuntimeError(
        f"Dependency installation failed - missing modules: {_missing}. "
        "Re-run this setup cell, or set USE_UV = False in Settings."
    )

# --- Optional: Flash Attention 2 (Ampere+ GPUs only) ---
if INSTALL_FLASH_ATTN:
    import subprocess as _fa_sp
    import re as _fa_re
    import urllib.request as _fa_url
    import json as _fa_json
    import torch

    if torch.cuda.is_available() and torch.cuda.get_device_capability(0)[0] >= 8:
        print(f"Installing Flash Attention 2 v{FLASH_ATTN_VERSION} via pre-built wheel...")
        _FA_VERSION = FLASH_ATTN_VERSION

        try:
            _cuda_raw = (torch.version.cuda or "").split(".")
            _cuda_major = int(_cuda_raw[0]) if _cuda_raw else 0
            _cuda_minor = int(_cuda_raw[1]) if len(_cuda_raw) > 1 else 0
            _cu_installed = _cuda_major * 10 + _cuda_minor

            _torch_ver = torch.__version__.split('+')[0]
            _torch_parts = _fa_re.sub(r'[^0-9.]', '', _torch_ver).split('.')
            _torch_parts = (_torch_parts + ['0', '0', '0'])[:3]
            _torch_major = int(_torch_parts[0])
            _torch_minor = int(_torch_parts[1])
            _th_installed = int(''.join(_torch_parts))

            _py_str = f"cp{sys.version_info.major}{sys.version_info.minor}"
            print(f"  Detected: CUDA {_cuda_major}.{_cuda_minor}, PyTorch {_torch_major}.{_torch_minor}.{_torch_parts[2]}, {_py_str}")

            _api_url = f"https://api.github.com/repos/Dao-AILab/flash-attention/releases/tags/v{_FA_VERSION}"
            _req = _fa_url.Request(_api_url, headers={"Accept": "application/vnd.github.v3+json"})
            with _fa_url.urlopen(_req, timeout=15) as _resp:
                _release = _fa_json.loads(_resp.read())

            _whl_re = _fa_re.compile(
                r"flash_attn-[\d.]+\+cu(\d+)torch(\d+)cxx11abiFALSE"
                r"-(" + _py_str + r")-\3-linux_x86_64\.whl"
            )
            _candidates = []
            for _asset in _release.get("assets", []):
                _m = _whl_re.match(_asset["name"])
                if _m:
                    _candidates.append((int(_m.group(1)), int(_m.group(2)), _asset["browser_download_url"], _asset["name"]))

            if not _candidates:
                print(f"  No pre-built wheels for {_py_str} in flash-attn v{_FA_VERSION} - falling back to SDPA")
            else:
                _compat = [c for c in _candidates if c[0] <= _cu_installed and c[1] <= _th_installed]
                if not _compat:
                    print("  No <=-compatible wheel found; trying highest available.")
                    _compat = sorted(_candidates, key=lambda x: (x[0], x[1]), reverse=True)
                else:
                    _compat.sort(key=lambda x: (x[0], x[1]), reverse=True)

                _fa_installed = False
                for _cu_n, _th_n, _dl_url, _whl_name in _compat:
                    print(f"  Trying: {_whl_name}")
                    try:
                        _r = _fa_sp.run([sys.executable, "-m", "pip", "install", _dl_url, "-q"],
                                        timeout=300, capture_output=True, text=True)
                        if _r.returncode == 0:
                            import flash_attn as _fa_mod
                            print(f"  Flash Attention 2 installed (v{_fa_mod.__version__})")
                            _fa_installed = True
                            break
                        else:
                            print(f"  Install failed (exit {_r.returncode}), trying next...")
                            if _r.stderr:
                                print(f"  {_r.stderr[:200]}")
                    except _fa_sp.TimeoutExpired:
                        print("  Install timed out after 300 s - trying next...")

                if not _fa_installed:
                    print("  All candidates failed - falling back to SDPA")
        except Exception as _e:
            print(f"  Could not install flash_attn ({_e}) - falling back to SDPA")
    else:
        print("Skipping flash_attn - only needed for Ampere+ GPUs (L4, A100, H100)")
# --- End flash_attn block ---

import torch
gpu_tier = "cpu"
if torch.cuda.is_available():
    cap = torch.cuda.get_device_capability(0)
    gpu_tier = "ampere" if cap[0] >= 8 else "turing"

config_path = os.path.expanduser('~/Qwen3-TTS_UserFiles/config.json')
with open(config_path) as f:
    config = json.load(f)

config['advanced']['backend'] = 'torch'
config['advanced']['model_size'] = MODEL_SIZE
config['advanced']['audio_loader'] = AUDIO_LOADER
config['models']['clone']['load_at_startup'] = PRELOAD_CLONE
config['models']['design']['load_at_startup'] = PRELOAD_DESIGN
config['models']['custom']['load_at_startup'] = PRELOAD_CUSTOM
config.setdefault('models', {}).setdefault('asr', {})['load_at_startup'] = PRELOAD_ASR
config['default_voice_description'] = DEFAULT_VOICE_DESC
config.setdefault('generation', {})['max_new_tokens'] = MAX_NEW_TOKENS
config['generation']['max_chunk_chars'] = MAX_CHUNK_CHARS
config['generation']['compile_model'] = COMPILE_MODEL


def _resolve_torch_quant(tier, override):
    if override and override != "auto":
        return override
    return "8bit" if tier == "turing" else "none"


if gpu_tier == "ampere":
    config['advanced']['dtype'] = 'bfloat16'
    config['advanced']['torch_quantization'] = _resolve_torch_quant("ampere", TORCH_QUANTIZATION)
    try:
        import flash_attn as _fa  # noqa: F401
        _attn_name = "Flash Attention 2"
    except ImportError:
        _attn_name = "SDPA"
    print(f'Ampere+ GPU detected - bfloat16, {_attn_name}, torch_quantization={config["advanced"]["torch_quantization"]}')
elif gpu_tier == "turing":
    config['advanced']['dtype'] = 'float16'
    config['advanced']['torch_quantization'] = _resolve_torch_quant("turing", TORCH_QUANTIZATION)
    print(f'Turing GPU detected - float16, SDPA, torch_quantization={config["advanced"]["torch_quantization"]}')
else:
    config['advanced']['dtype'] = 'float32'
    config['advanced']['torch_quantization'] = 'none'
    print('No GPU detected - float32 (will be slow)')

with open(config_path, 'w') as f:
    json.dump(config, f, indent=2)

print(f'Model size: {MODEL_SIZE}')
print(f'Preload: clone={PRELOAD_CLONE}, design={PRELOAD_DESIGN}, custom={PRELOAD_CUSTOM}, asr={PRELOAD_ASR}')
print(f'Max new tokens: {MAX_NEW_TOKENS}, Max chunk chars: {MAX_CHUNK_CHARS}')
print(f'Compile model: {COMPILE_MODEL}, Audio loader: {AUDIO_LOADER}')

print(f'\nCUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

print('\nSetup complete!')

In [ ]:
# === Cell 2: Start TTS Server ===
import subprocess
import time
import requests
import sys
import os

!kill $(lsof -t -i:5123) 2>/dev/null || true

err_log = os.path.expanduser('~/server_stderr.log')
with open(err_log, 'w') as ef:
    server = subprocess.Popen(
        ['python', '-m', 'qwen3_tts.server.app'],
        cwd=os.path.expanduser('~/Qwen3-TTS_UserFiles'),
        env={**os.environ, 'PYTHONPATH': os.path.expanduser('~/Qwen3-TTS_UserFiles')},
        stdout=subprocess.PIPE, stderr=ef
    )


def _dump_pip_diagnostics():
    """Print package versions for the modules most likely to be missing."""
    print('\n--- Installed package versions (for triage) ---')
    import subprocess as _sp
    _critical = ["slowapi", "psutil", "pyloudnorm", "rich", "fastapi",
                 "uvicorn", "transformers", "torch", "qwen-tts", "gradio"]
    try:
        _r = _sp.run(["pip", "show", *_critical], capture_output=True, text=True, timeout=15)
        for _line in _r.stdout.split("\n"):
            if _line.startswith(("Name:", "Version:")):
                print(_line)
    except Exception as _e:
        print(f"  pip show failed: {_e}")
    print('--- End diagnostics ---\n')


server_url = 'http://127.0.0.1:5123'
print('Starting server...', end='')
for i in range(180):
    if server.poll() is not None:
        print(f'\nServer exited with code {server.returncode}')
        with open(err_log) as f:
            print(f.read()[-3000:])
        _dump_pip_diagnostics()
        raise RuntimeError('Server failed to start - check errors above')
    try:
        resp = requests.get(f'{server_url}/health', timeout=1)
        if resp.status_code == 200:
            print(f'\nServer ready! (took {i+1}s)')
            health = resp.json()
            print(f'  Backend: {health.get("backend", "N/A")}')
            print(f'  Model size: {health.get("model_size", "N/A")}')
            break
    except requests.ConnectionError:
        pass
    print('.', end='', flush=True)
    time.sleep(1)
else:
    print('\nServer did not respond after 180s. Stderr:')
    with open(err_log) as f:
        print(f.read()[-3000:])
    _dump_pip_diagnostics()

import torch
if torch.cuda.is_available():
    allocated = torch.cuda.memory_allocated() / 1e9
    total = torch.cuda.get_device_properties(0).total_memory / 1e9
    free = total - allocated
    print(f'\nGPU Memory: {allocated:.1f} GB used / {total:.1f} GB total ({free:.1f} GB free)')

In [ ]:
# === Cell 3: Launch Gradio UI ===
# Access the UI via the public URL printed below.
import os
import tempfile
if AUTO_LAUNCH_UI:
    from qwen3_tts.interface.ui import build_ui
    demo = build_ui()
    demo.launch(
        server_name='0.0.0.0',
        share=True,
        allowed_paths=[os.path.expanduser('~/Downloads'), tempfile.gettempdir()],
    )
else:
    print('Gradio UI skipped (AUTO_LAUNCH_UI = False in Settings).')
    print('You can still generate audio using the TTSClient in the cells below.')

In [ ]:
# === Cell 4: Quick Generation Example (without UI) ===
from qwen3_tts.server.client import TTSClient

client = TTSClient()
output = client.generate(
    'Hello from Google Colab! This is Qwen3 TTS running on a GPU.',
    mode=DEFAULT_MODE,
    description=DEFAULT_VOICE_DESC,
    output='colab_test.wav'
)
print(f'Generated: {output}')

# Play in notebook
from IPython.display import Audio
Audio(output)

# Waveform visualization
try:
    import soundfile as sf
    import matplotlib.pyplot as plt
    import numpy as np
    wav, sr = sf.read(output)
    t = np.arange(len(wav)) / sr
    fig, ax = plt.subplots(figsize=(10, 2))
    ax.plot(t, wav, linewidth=0.3)
    ax.set_xlabel('Time (s)')
    ax.set_ylabel('Amplitude')
    ax.set_title('Generated Waveform')
    plt.tight_layout()
    plt.show()
except ImportError:
    pass

In [ ]:
# === Cell 5: Model Management ===
from qwen3_tts.server.client import TTSClient

client = TTSClient()
models = client.get_models()

print('Model Status:')
for name, info in models.get('models', {}).items():
    status = 'LOADED' if info.get('loaded') else 'unloaded'
    mem = f" ({info.get('memory_mb', '?')} MB)" if info.get('loaded') else ''
    print(f'  {name}: {status}{mem}')

print(f'\nBackend: {models.get("backend", "N/A")}')
print(f'Model size: {models.get("model_size", "N/A")}')

# Uncomment to load/unload models on demand:
# client.load_model('design')    # Load the design model
# client.load_model('custom')    # Load the custom model
# client.unload_model('clone')   # Free clone model memory

In [ ]:
# === Cell 6: Create Voice Clone === #@title Voice Cloning { display-mode: "form" }
VOICE_NAME = "my_voice"  # @param {type:"string"}
TRANSCRIPT = ""  # @param {type:"string"}

# Upload audio file
from google.colab import files
print('Upload a .wav or .mp3 audio file (5-30 seconds of clear speech):')
uploaded = files.upload()

if uploaded:
    audio_file = list(uploaded.keys())[0]
    print(f'Uploaded: {audio_file}')

    # Ensure clone model is loaded
    from qwen3_tts.server.client import TTSClient
    client = TTSClient()
    try:
        client.load_model('clone')
    except Exception:
        pass

    # Create voice prompt
    from qwen3_tts.tools.create_voice import create_and_save_voice_prompt

    create_and_save_voice_prompt(
        audio_path=audio_file,
        prompt_name=VOICE_NAME,
        transcript=TRANSCRIPT or None,
    )
    print(f'\nVoice prompt created: {VOICE_NAME}')
    print('Use it with: client.generate("text", mode="clone", prompt=f"{VOICE_NAME}.pt")')
else:
    print('No file uploaded.')

In [ ]:
# === Cell 7: Troubleshooting ===
# Run this cell if you encounter errors to see the server log.
with open(os.path.expanduser('~/server_stderr.log')) as f:
    print(f.read()[-3000:])